In [1]:
import pandas as pd

df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')
df.head(5)

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [3]:
# 1. Check data types, column names, and non-null counts
print('--- DATASET STRUCTURE ---')
df.info()


--- DATASET STRUCTURE ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14

In [7]:
# 2. Check statistical features for numerical columns (mean, min, max, quartiles)
print('\n--- NUMERICAL FEATURES SUMMARY ---')
print(df.describe().round(2))


--- NUMERICAL FEATURES SUMMARY ---
           Age  DailyRate  DistanceFromHome  Education  EmployeeCount  \
count  1470.00    1470.00           1470.00    1470.00         1470.0   
mean     36.92     802.49              9.19       2.91            1.0   
std       9.14     403.51              8.11       1.02            0.0   
min      18.00     102.00              1.00       1.00            1.0   
25%      30.00     465.00              2.00       2.00            1.0   
50%      36.00     802.00              7.00       3.00            1.0   
75%      43.00    1157.00             14.00       4.00            1.0   
max      60.00    1499.00             29.00       5.00            1.0   

       EmployeeNumber  EnvironmentSatisfaction  HourlyRate  JobInvolvement  \
count         1470.00                  1470.00     1470.00         1470.00   
mean          1024.87                     2.72       65.89            2.73   
std            602.02                     1.09       20.33            0.

In [8]:
# 3. Check summary for text/categorical columns (top frequent values, unique counts)
print('\n--- CATEGORICAL FEATURES SUMMARY ---')
print(df.describe(include='object'))


--- CATEGORICAL FEATURES SUMMARY ---
       Attrition BusinessTravel              Department EducationField Gender  \
count       1470           1470                    1470           1470   1470   
unique         2              3                       3              6      2   
top           No  Travel_Rarely  Research & Development  Life Sciences   Male   
freq        1233           1043                     961            606    882   

                JobRole MaritalStatus Over18 OverTime  
count              1470          1470   1470     1470  
unique                9             3      1        2  
top     Sales Executive       Married      Y       No  
freq                326           673   1470     1054  


In [ ]:
# PIVOT TABLES
# Filter only active employees first using query()
active_df = df.query("Attrition == 'No'")

# Create a cross-tabulation (9-Box Grid preview)
talent_grid = pd.crosstab(
    index=active_df['PerformanceRating'],
    columns=active_df['JobInvolvement'],
    margins=True,  # Adds 'All' row and column totals
    margins_name='Total',
)

print('\n--- 9-BOX GRID PREVIEW (ACTIVE STAFF) ---')
print(talent_grid)



--- 9-BOX GRID PREVIEW (ACTIVE STAFF) ---
JobInvolvement      1    2    3    4  Total
PerformanceRating                          
3                  42  259  631  112   1044
4                  13   45  112   19    189
Total              55  304  743  131   1233


In [11]:
# Pivot table showing average years since promotion & training by Department and JobLevel
dev_pivot = pd.pivot_table(
    data=active_df,
    values=['YearsSinceLastPromotion', 'TrainingTimesLastYear'],
    index=['Department', 'JobLevel'],
    aggfunc='mean',
).round(2)

print('\n--- DEVELOPMENT METRICS BY DEPARTMENT & LEVEL ---')
print(dev_pivot)



--- DEVELOPMENT METRICS BY DEPARTMENT & LEVEL ---
                                 TrainingTimesLastYear  \
Department             JobLevel                          
Human Resources        1                          2.87   
                       2                          2.62   
                       3                          2.00   
                       4                          3.00   
                       5                          2.29   
Research & Development 1                          2.84   
                       2                          2.78   
                       3                          2.95   
                       4                          2.58   
                       5                          3.02   
Sales                  1                          3.02   
                       2                          2.88   
                       3                          2.95   
                       4                          2.43   
                     

In [12]:
# Count of active employees per Department across Job Levels (1 to 5)
pipeline_pivot = pd.pivot_table(
    data=active_df,
    values='EmployeeNumber',
    index='Department',
    columns='JobLevel',
    aggfunc='count',
    fill_value=0,  # Replaces empty cells with 0
    margins=True,
)

print('\n--- SUCCESSION PIPELINE HEADCOUNT ---')
print(pipeline_pivot)



--- SUCCESSION PIPELINE HEADCOUNT ---
JobLevel                  1    2    3    4   5   All
Department                                          
Human Resources          23   13    4    4   7    51
Research & Development  333  266  116   67  46   828
Sales                    44  203   66   30  11   354
All                     400  482  186  101  64  1233



FINDING AND CONCLUSION
1. Dataset Health & Structure (df.info & df.describe)

• Total Records: 1,470 employees across 35 columns (1,233 active employees and 237 resigned).
• Data Completeness: 0 missing values and 0 duplicate IDs, meaning the dataset requires minimal cleaning before loading into Power BI.
• Constant Columns to Drop: Three columns (EmployeeCount, Over18, and StandardHours) contain only a single constant value across all rows and can be safely removed to keep your Power BI data model lightweight.

2. Key People Development Findings
A. The Performance Rating Skew (PerformanceRating)

• Finding: All 1,470 employees in the dataset have a PerformanceRating of either 3 (Excellent = 84.6%) or 4 (Outstanding = 15.4%). There are no ratings of 1 or 2.
• Dashboard Impact: A standard 3x3 9-Box Grid using raw PerformanceRating will have an empty bottom row. To make the dashboard matrix meaningful, need to combine PerformanceRating with PercentSalaryHike or JobSatisfaction to create a 3-tier performance score (Developing, Solid, Top).

B. Training Distribution (TrainingTimesLastYear)

• Finding: Employees receive an average of 2.8 training sessions per year (ranging from 0 to 6). Most staff cluster around 2 or 3 sessions, regardless of whether they are high or average performers.
• Dashboard Impact: There is no strong direct correlation between training frequency and higher PerformanceRating or lower Attrition. This highlights a great business insight for HR: training is currently distributed uniformly rather than targeted at skill gaps or high-potential talent.

C. Career Stagnation Bottlenecks (YearsSinceLastPromotion)

• Finding: While the overall average time since last promotion is 2.19 years, senior levels (JobLevel 3, 4, and 5) experience much longer wait times—often 4 to 7+ years without a promotion.
• Dashboard Impact: Stagnation is heavily concentrated in mid-to-senior roles within Research & Development and Sales. Filtering for active employees with PerformanceRating == 4 and YearsSinceLastPromotion >= 4 immediately isolates your highest flight-risk talent.

D. Leadership Pipeline Shape (JobLevel)

• Finding: The organizational structure forms a steep pyramid:
	• Level 1 & 2 (Staff / Senior Staff): ~73% of the workforce (1,077 employees).
	• Level 3 (Manager): ~15% (218 employees).
	• Level 4 & 5 (Director / Executive): ~12% (175 employees).
• Dashboard Impact: There is a steep drop-off between Level 2 and Level 3, making the Supervisor-to-Manager transition the most critical succession bottleneck to monitor in the Power BI pipeline visual.
